# AGILAB worker class paths in Colab (PyPI)

This notebook separates worker-class resolution from the app launch flow for the published-package path.

- `execution_pandas_project` resolves to `ExecutionPandasWorker` and the `PandasWorker` family.
- `flight_project` resolves to `FlightWorker` and the `PolarsWorker` family.
- `uav_relay_queue_project` is shown through `AgiEnv` path resolution, while `DagWorker` is shown separately from the shared core.

It installs AGILAB from PyPI and inspects the packaged built-in app paths directly.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_worker_paths_pypi.ipynb)


In [ ]:
# Published-package Colab path: install AGILAB from PyPI.
import json
import os
import subprocess
import sys
from pathlib import Path

STATE_FILE = Path("/content/.agilab_colab_pypi_state.json")
STATE_VERSION = 4
SENSITIVE_PREFIXES = (
    "agilab.",
    "agi_env", "agi_env.",
    "agi_node", "agi_node.",
    "agi_cluster", "agi_cluster.",
    "numpy", "numpy.",
    "scipy", "scipy.",
    "pandas", "pandas.",
    "polars", "polars.",
    "numba", "numba.",
)
state = {}
if STATE_FILE.is_file():
    try:
        state = json.loads(STATE_FILE.read_text())
    except Exception:
        state = {}

needs_install = (
    state.get("version") != STATE_VERSION
    or state.get("mode") != "pypi"
)

if needs_install:
    restart_required = any(name == "agilab" or name.startswith(SENSITIVE_PREFIXES) for name in sys.modules)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "agilab", "agi-core", "agi-cluster", "agi-node", "agi-env"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-cache-dir",
        "uv", "agi-core", "agilab",
    ], check=True)
    STATE_FILE.write_text(json.dumps({
        "version": STATE_VERSION,
        "mode": "pypi",
        "install_pid": os.getpid(),
        "restart_required": restart_required,
    }))
    if restart_required:
        raise SystemExit("AGILAB PyPI packages were updated after scientific or AGILAB modules were already loaded. Restart the Colab runtime; if the browser shows a temporary disconnect, reconnect to the session, then rerun the notebook from the top.")
    print("AGILAB PyPI packages installed in a fresh Colab kernel; continuing without restart.")
else:
    print("AGILAB PyPI packages already prepared for this Colab runtime.")


In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

STATE_FILE = Path("/content/.agilab_colab_pypi_state.json")
if STATE_FILE.is_file():
    try:
        _colab_state = json.loads(STATE_FILE.read_text())
    except Exception:
        _colab_state = {}
    if _colab_state.get("install_pid") == os.getpid() and _colab_state.get("restart_required"):
        raise RuntimeError("AGILAB PyPI packages were installed in this same Colab kernel after scientific or AGILAB modules were already loaded. Restart the runtime; if the browser shows a temporary disconnect, reconnect to the session, then rerun the notebook from the top.")

for bad_prefix in ("/content/agilab/src", "/content/agilab"):
    sys.path = [p for p in sys.path if p != bad_prefix and not p.startswith(bad_prefix + "/")]
for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

import agilab
from agi_env import AgiEnv
from agi_node.dag_worker import DagWorker
from agi_node.pandas_worker import PandasWorker
from agi_node.polars_worker import PolarsWorker

BUILTIN_ROOT = Path(agilab.__file__).resolve().parent / "apps" / "builtin"
for entry in reversed([
    BUILTIN_ROOT / "execution_pandas_project" / "src",
    BUILTIN_ROOT / "flight_project" / "src",
]):
    entry_str = str(entry)
    if entry_str not in sys.path:
        sys.path.insert(0, entry_str)

print("Builtin root:", BUILTIN_ROOT)
print("execution_pandas_project source:", (BUILTIN_ROOT / "execution_pandas_project" / "src").is_dir())
print("flight_project source:", (BUILTIN_ROOT / "flight_project" / "src").is_dir())
print("uav_relay_queue_project source:", (BUILTIN_ROOT / "uav_relay_queue_project" / "src").is_dir())

from execution_pandas_worker.execution_pandas_worker import ExecutionPandasWorker
from flight_worker.flight_worker import FlightWorker

def describe_app(app_name: str, imported_cls=None):
    env = AgiEnv(apps_path=BUILTIN_ROOT, app=app_name, verbose=0)
    print(f"\n=== {app_name} ===")
    print("target worker class:", getattr(env, "target_worker_class", None))
    print("worker source:", getattr(env, "worker_path", None))
    print("resolved worker target:", getattr(env, "target_worker", None))
    if imported_cls is not None:
        print("imported class:", imported_cls.__module__ + "." + imported_cls.__name__)
        print("MRO:", " -> ".join(cls.__name__ for cls in imported_cls.mro()[:4]))
    return env

describe_app("execution_pandas_project", ExecutionPandasWorker)
describe_app("flight_project", FlightWorker)
describe_app("uav_relay_queue_project")

print("\n=== shared DagWorker ===")
print("imported class:", DagWorker.__module__ + "." + DagWorker.__name__)
print("MRO:", " -> ".join(cls.__name__ for cls in DagWorker.mro()[:4]))
print("base worker families:", PandasWorker.__name__, "/", PolarsWorker.__name__)


## What this proves

- The published AGILAB package exposes the built-in app paths needed for worker-class inspection.
- `AgiEnv` resolves the worker path for each packaged built-in app.
- `execution_pandas_project` binds to `ExecutionPandasWorker`, whose base family is `PandasWorker`.
- `flight_project` binds to `FlightWorker`, whose base family is `PolarsWorker`.
- `DagWorker` is available as the shared DAG execution base class.
- `uav_relay_queue_project` stays visible as a resolved app worker path, without pretending it is the same thing as `DagWorker`.

## Next step

After this class-path check works, use the launch notebooks if you want to run the apps end to end.
